In [47]:
import json
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.signal import find_peaks, savgol_filter
from typing import Pattern

import utils

In [9]:
OUTPUTS_DIR = Path("/Users/hyeon/SeSAC_Project/PoC/run/test7/outputs")
hpe = OUTPUTS_DIR / "pose_predictions.json"

with hpe.open(mode= "r", encoding= 'utf-8') as file:
    pose_data = json.load(file)

details = OUTPUTS_DIR / "details.json"
with details.open(mode= "r", encoding= 'utf-8') as file:
    detail_data = json.load(file)

In [ ]:
class PoseSequence:
    def __init__(self, pose_data: dict, detail_data: dict):
        # bbox, keypoints들의 y좌표 반전.
        df = utils.hpe2pd(pose_data)
        _cols = df.columns[df.columns.str.contains("_y|up|down")]
        df.loc[:, _cols] = detail_data["video"]["height"] - df[_cols]

        # direction
        dir_index = 0 if (df["nose_x"] - df["neck_x"])[0] > 0 else 1

        self.df = df
        self.details = detail_data
        self.direction = "left" if dir_index else "right"
        self.strides = []

    def cal_stride(self) -> None:
        """
        self.stride에 1스트라이드의 시작, 끝 프레임 기록
        """
        df = self.df
        dir = self.direction
        left_knee_angle = utils.knee_flexion_angle(
            [df[f"{dir}_hip_x"], df[f"{dir}_hip_y"]],
            [df[f"{dir}_knee_x"], df[f"{dir}_knee_y"]],
            [df[f"{dir}_ankle_x"], df[f"{dir}_ankle_y"]]
        )
        y = np.asarray(left_knee_angle, dtype=float)
        y_smooth = savgol_filter(
            y,
            window_length=5,
            polyorder=2
        )
        # 데이터 범위에 비례해 prominence 설정
        signal_range = np.percentile(y_smooth, 95) - np.percentile(y_smooth, 5)
        prominence = signal_range * 0.10    # 곱해지는 숫자가 클수록 큰 봉우리만 검출

        # 최댓값
        max_indices, max_properties = find_peaks(
            y_smooth,
            prominence=prominence,
            distance=10,
        )

        # 최솟값: 신호에 -를 붙여서 봉우리로 변환
        min_indices, min_properties = find_peaks(
            -y_smooth,
            prominence=prominence,
            distance=10,
        )

        _max = {
            "frame": max_indices,
            "class": np.ones_like(max_indices)
        }
        _min = {
            "frame": min_indices,
            "class": np.zeros_like(min_indices)
        }
        extremum = pd.concat([pd.DataFrame(_max), pd.DataFrame(_min)]).sort_values('frame')

        assert len(extremum) >= 2, "온전한 스트라이드가 검출되지 않았습니다."
        # 올바른 스트라이드 검출
        for i in range(1, len(extremum)):
            previous = extremum["class"].iloc[i - 1]
            current = extremum["class"].iloc[i]

            assert {previous, current} == {0, 1}, (
                f"예외: {i-1}, {i}행의 값이 {previous}, {current}입니다.\n올바른 스트라이드가 검출되지 않았습니다. 검출 파라미터 변경 필요."
            )
        for i in range(len(extremum) // 4 + 1):
            self.strides.append(extremum['frame'].iloc[4 * i : 4 * i + 4].to_list())
    def gct(self, next: int = 0):
        """
        {side}의 heel이 지면에 접촉하고 big_toe가 지면에서 떼어지는 순간까지의 인덱스 출력
        """
        n = len(self.strides) - (0 if len(self.strides[-1]) == 5 else 1)
        res = []
        for i in range(n):
            start = self.strides[n][0]
            end = self.strides[n][-1]

            df = self.df.loc[start:end+1]
            heel = f"{self.direction}_heel"
            td = int(df[f"{heel}_y"].argmin())

            toe = f"{self.direction}_big_toe"
            min_value = df[f"{toe}_y"].min()

            inside = df[f"{toe}_y"].between(min_value, min_value + 5)
            to = df.index[~inside & inside.shift(1, fill_value=False)].to_list()[0]

            res.append([td, to])
        return res